# 2017 Finals Castles In The Air - Data Analysis Solution

Model property investments monthly, compute NPV using XNPV, and select optimal portfolio.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, date
from dateutil.relativedelta import relativedelta
import os

In [ ]:
# ============================================================
# Helper: XNPV function
# ============================================================
def xnpv(rate, cashflows, dates):
    """
    Calculate XNPV like Excel's XNPV function.
    rate: annual discount rate (e.g. 0.08 for 8%)
    cashflows: list/array of cash flows
    dates: list/array of datetime.date objects
    The first date is the valuation date.
    """
    d0 = dates[0]
    total = 0.0
    for cf, d in zip(cashflows, dates):
        days = (d - d0).days
        total += cf / ((1.0 + rate) ** (days / 365.0))
    return total

In [ ]:
# ============================================================
# Helper: end of month date
# ============================================================
import calendar

def end_of_month(year, month):
    """Return the last day of the given month."""
    day = calendar.monthrange(year, month)[1]
    return date(year, month, day)

In [ ]:
# ============================================================
# Generate monthly dates for an investment
# Purchase date: 31 Dec 2017
# Monthly model from Jan 2018 onwards
# ============================================================
purchase_date = date(2017, 12, 31)

def generate_monthly_dates(investment_years):
    """
    Generate end-of-month dates from Jan 2018 for the given number of years.
    investment_years can be fractional (e.g. 5.5 = 5 years 6 months = 66 months).
    """
    n_months = int(round(investment_years * 12))
    dates_list = []
    for i in range(1, n_months + 1):
        y = 2018 + (i - 1) // 12
        m = ((i - 1) % 12) + 1
        dates_list.append(end_of_month(y, m))
    return dates_list

In [ ]:
# ============================================================
# Indexation helper
# Base date: 1 Jan 2018
# Index steps annually: first full year of indexation on 1 Jan 2019
# For a date in year Y (Jan-Dec Y):
#   2018: factor = 1.0 (base year)
#   2019: factor = (1 + rate)^1
#   2020: factor = (1 + rate)^2
#   etc.
# ============================================================
def index_factor(d, annual_rate):
    """Return the indexation factor for a given date."""
    year = d.year
    years_from_base = year - 2018
    if years_from_base <= 0:
        return 1.0
    return (1.0 + annual_rate) ** years_from_base

## Property 1
- Purchase Price: $450,000
- Investment length: 5 years (60 months: Jan 2018 - Dec 2022)
- Terminal value: $500,000 (received at end of investment)
- Rental revenue: $45,000/year, paid monthly, indexed at 2.5%
- Operating costs: 5% of revenues

In [ ]:
# ============================================================
# Property 1
# ============================================================
p1_purchase = 450000
p1_investment_years = 5
p1_terminal = 500000
p1_annual_revenue = 45000
p1_index_rate = 0.025

p1_dates = generate_monthly_dates(p1_investment_years)

# Revenue: $45,000/year paid monthly = $3,750/month base, indexed at 2.5%
p1_monthly_base = p1_annual_revenue / 12.0

p1_revenues = []
p1_costs = []
for d in p1_dates:
    factor = index_factor(d, p1_index_rate)
    rev = p1_monthly_base * factor
    cost = rev * 0.05  # 5% of revenues
    p1_revenues.append(rev)
    p1_costs.append(cost)

p1_total_revenues = sum(p1_revenues)
p1_total_costs = sum(p1_costs)
print(f"Property 1 total revenues: ${p1_total_revenues:,.2f}")
print(f"Property 1 total costs: ${p1_total_costs:,.2f}")

In [ ]:
# Question 1: What are the total revenues for Property 1?
# Expected answer: H ($236,535)
print(f"Q1: Property 1 total revenues = ${p1_total_revenues:,.0f}")
# Match to multiple choice
q1_options = {'A': 236528, 'B': 236529, 'C': 236530, 'D': 236531,
              'E': 236532, 'F': 236533, 'G': 236534, 'H': 236535, 'I': 236536}
q1_answer = min(q1_options.keys(), key=lambda k: abs(q1_options[k] - round(p1_total_revenues)))
print(f"Q1 answer: {q1_answer}")

## Property 2
- Purchase Price: $550,000
- Investment length: 5.5 years (66 months: Jan 2018 - Jun 2023)
- Terminal value: $575,000
- Rental revenue: $60,000/year, paid quarterly (starting March), indexed at 3%
- Operating costs: $4,500/year, paid monthly, indexed at 3%

In [ ]:
# ============================================================
# Property 2
# ============================================================
p2_purchase = 550000
p2_investment_years = 5.5
p2_terminal = 575000
p2_annual_revenue = 60000
p2_index_rate = 0.03
p2_annual_opcost = 4500

p2_dates = generate_monthly_dates(p2_investment_years)

# Revenue: $60,000/year paid quarterly starting March
# Quarterly = $15,000 per quarter base
# Paid in March, June, September, December
p2_quarterly_base = p2_annual_revenue / 4.0

p2_revenues = []
p2_costs = []
p2_monthly_opcost_base = p2_annual_opcost / 12.0

for d in p2_dates:
    factor = index_factor(d, p2_index_rate)
    # Revenue: only in March, June, September, December
    if d.month in [3, 6, 9, 12]:
        rev = p2_quarterly_base * factor
    else:
        rev = 0.0
    # Operating costs: monthly, indexed at 3%
    cost = p2_monthly_opcost_base * factor
    p2_revenues.append(rev)
    p2_costs.append(cost)

p2_total_revenues = sum(p2_revenues)
p2_total_costs = sum(p2_costs)
print(f"Property 2 total revenues: ${p2_total_revenues:,.2f}")
print(f"Property 2 total costs: ${p2_total_costs:,.2f}")

In [ ]:
# Question 2: What are the revenues for Property 2 in September 2019?
# In Sep 2019, it's a quarterly payment month.
# Index factor: year 2019 => (1.03)^1 = 1.03
# Revenue = 15000 * 1.03 = 15450
sep2019 = end_of_month(2019, 9)
p2_rev_sep2019 = p2_quarterly_base * index_factor(sep2019, p2_index_rate)
print(f"Q2: Property 2 revenue in Sep 2019 = ${p2_rev_sep2019:,.2f}")

q2_options = {'A': 15446, 'B': 15447, 'C': 15448, 'D': 15449,
              'E': 15450, 'F': 15451, 'G': 15452, 'H': 15453, 'I': 15454}
q2_answer = min(q2_options.keys(), key=lambda k: abs(q2_options[k] - round(p2_rev_sep2019)))
print(f"Q2 answer: {q2_answer}")

## Property 3
- Purchase Price: $500,000
- Investment length: 6 years (72 months: Jan 2018 - Dec 2023)
- Terminal value: $550,000
- Rental revenue: $55,000/year, paid quarterly (starting January), indexed at 2%
- Operating costs: $1,000 in April, $3,000 in October, indexed at 2%

In [ ]:
# ============================================================
# Property 3
# ============================================================
p3_purchase = 500000
p3_investment_years = 6
p3_terminal = 550000
p3_annual_revenue = 55000
p3_index_rate = 0.02

p3_dates = generate_monthly_dates(p3_investment_years)

# Revenue: $55,000/year paid quarterly starting January
# Quarterly = $13,750 per quarter base
# Paid in January, April, July, October
p3_quarterly_base = p3_annual_revenue / 4.0

p3_revenues = []
p3_costs = []

for d in p3_dates:
    factor = index_factor(d, p3_index_rate)
    # Revenue: only in Jan, Apr, Jul, Oct
    if d.month in [1, 4, 7, 10]:
        rev = p3_quarterly_base * factor
    else:
        rev = 0.0
    # Operating costs: $1,000 in April, $3,000 in October, indexed at 2%
    if d.month == 4:
        cost = 1000.0 * factor
    elif d.month == 10:
        cost = 3000.0 * factor
    else:
        cost = 0.0
    p3_revenues.append(rev)
    p3_costs.append(cost)

p3_total_revenues = sum(p3_revenues)
p3_total_costs = sum(p3_costs)
print(f"Property 3 total revenues: ${p3_total_revenues:,.2f}")
print(f"Property 3 total costs: ${p3_total_costs:,.2f}")

In [ ]:
# Question 3: What are the costs for Property 3 in October 2020?
# October 2020: index factor = (1.02)^2 = 1.0404
# Cost = 3000 * 1.0404 = 3121.2
oct2020 = end_of_month(2020, 10)
p3_cost_oct2020 = 3000.0 * index_factor(oct2020, p3_index_rate)
print(f"Q3: Property 3 cost in Oct 2020 = ${p3_cost_oct2020:,.2f}")

q3_options = {'A': 3118, 'B': 3119, 'C': 3120, 'D': 3121,
              'E': 3122, 'F': 3123, 'G': 3124, 'H': 3125, 'I': 3126}
q3_answer = min(q3_options.keys(), key=lambda k: abs(q3_options[k] - round(p3_cost_oct2020)))
print(f"Q3 answer: {q3_answer}")

## Property 4a (without overhaul)
- Purchase Price: $470,000
- Investment length: 4 years (48 months: Jan 2018 - Dec 2021)
- Terminal value: $570,000
- Rental revenue: $55,000/year, paid monthly, NOT indexed
- Operating costs: $3,000/year, paid monthly, indexed at 1%

In [ ]:
# ============================================================
# Property 4a
# ============================================================
p4a_purchase = 470000
p4a_investment_years = 4
p4a_terminal = 570000
p4a_annual_revenue = 55000
p4a_annual_opcost = 3000
p4a_opcost_index = 0.01

p4a_dates = generate_monthly_dates(p4a_investment_years)

p4a_monthly_rev = p4a_annual_revenue / 12.0
p4a_monthly_opcost_base = p4a_annual_opcost / 12.0

p4a_revenues = []
p4a_costs = []

for d in p4a_dates:
    # Revenue: monthly, NOT indexed
    rev = p4a_monthly_rev
    # Operating costs: monthly, indexed at 1%
    factor = index_factor(d, p4a_opcost_index)
    cost = p4a_monthly_opcost_base * factor
    p4a_revenues.append(rev)
    p4a_costs.append(cost)

p4a_total_revenues = sum(p4a_revenues)
p4a_total_costs = sum(p4a_costs)
p4a_net = p4a_total_revenues - p4a_total_costs
print(f"Property 4a total revenues: ${p4a_total_revenues:,.2f}")
print(f"Property 4a total costs: ${p4a_total_costs:,.2f}")
print(f"Property 4a revenues less costs: ${p4a_net:,.2f}")

In [ ]:
# Question 4: What are total revenues less total costs for Property 4a?
print(f"Q4: Property 4a rev - costs = ${p4a_net:,.0f}")

q4_options = {'A': 207819, 'B': 207820, 'C': 207821, 'D': 207822,
              'E': 207823, 'F': 207824, 'G': 207825, 'H': 207826, 'I': 207827}
q4_answer = min(q4_options.keys(), key=lambda k: abs(q4_options[k] - round(p4a_net)))
print(f"Q4 answer: {q4_answer}")

## Property 4b (with overhaul)
- Purchase Price: same as 4a ($470,000)
- Investment length: same as 4a (4 years)
- Overhaul cost: $125,000 paid 31 Dec 2019
- Terminal value: $675,000
- Rental revenue: Up to overhaul same as 4a; afterwards $75,000/year paid monthly, not indexed
- Operating costs: Up to overhaul same as 4a; afterwards 8% of revenues

In [ ]:
# ============================================================
# Property 4b
# ============================================================
p4b_purchase = 470000  # same as 4a
p4b_investment_years = 4  # same as 4a
p4b_terminal = 675000
p4b_overhaul_cost = 125000
p4b_overhaul_date = date(2019, 12, 31)
p4b_post_annual_revenue = 75000

p4b_dates = generate_monthly_dates(p4b_investment_years)

p4b_revenues = []
p4b_costs = []

for d in p4b_dates:
    if d <= p4b_overhaul_date:
        # Before/during overhaul: same as 4a
        rev = p4a_monthly_rev  # $55,000/12/month, not indexed
        factor = index_factor(d, p4a_opcost_index)
        cost = p4a_monthly_opcost_base * factor
    else:
        # After overhaul: $75,000/year paid monthly, not indexed
        rev = p4b_post_annual_revenue / 12.0
        cost = rev * 0.08  # 8% of revenues
    p4b_revenues.append(rev)
    p4b_costs.append(cost)

p4b_total_revenues = sum(p4b_revenues)
p4b_total_costs = sum(p4b_costs)
print(f"Property 4b total revenues: ${p4b_total_revenues:,.2f}")
print(f"Property 4b total costs: ${p4b_total_costs:,.2f}")

In [ ]:
# Question 5: absolute value of difference in operating costs between 4a and 4b
q5_diff = abs(p4a_total_costs - p4b_total_costs)
print(f"Q5: |4a costs - 4b costs| = ${q5_diff:,.2f}")

q5_options = {'A': 5848, 'B': 5849, 'C': 5850, 'D': 5851,
              'E': 5852, 'F': 5853, 'G': 5854, 'H': 5855, 'I': 5856}
q5_answer = min(q5_options.keys(), key=lambda k: abs(q5_options[k] - round(q5_diff)))
print(f"Q5 answer: {q5_answer}")

In [ ]:
# ============================================================
# NPV Calculations for all properties at 8%
# ============================================================
rate = 0.08

def compute_npv(purchase, dates_list, revenues, costs, terminal, rate,
                overhaul_cost=0, overhaul_date=None):
    """
    Compute NPV for a property investment.
    Purchase price paid on 31 Dec 2017.
    Terminal value received at end of last month.
    """
    all_dates = [purchase_date]  # purchase date
    all_cfs = [-purchase]       # purchase cost (outflow)
    
    for i, d in enumerate(dates_list):
        net_cf = revenues[i] - costs[i]
        # Add overhaul cost if applicable
        if overhaul_date and d == overhaul_date:
            net_cf -= overhaul_cost
        # Add terminal value at last date
        if i == len(dates_list) - 1:
            net_cf += terminal
        all_dates.append(d)
        all_cfs.append(net_cf)
    
    return xnpv(rate, all_cfs, all_dates)

# Property 1 NPV
npv_p1 = compute_npv(p1_purchase, p1_dates, p1_revenues, p1_costs, p1_terminal, rate)
print(f"NPV Property 1: ${npv_p1:,.2f}")

# Property 2 NPV
npv_p2 = compute_npv(p2_purchase, p2_dates, p2_revenues, p2_costs, p2_terminal, rate)
print(f"NPV Property 2: ${npv_p2:,.2f}")

# Property 3 NPV
npv_p3 = compute_npv(p3_purchase, p3_dates, p3_revenues, p3_costs, p3_terminal, rate)
print(f"NPV Property 3: ${npv_p3:,.2f}")

# Property 4a NPV
npv_p4a = compute_npv(p4a_purchase, p4a_dates, p4a_revenues, p4a_costs, p4a_terminal, rate)
print(f"NPV Property 4a: ${npv_p4a:,.2f}")

# Property 4b NPV
npv_p4b = compute_npv(p4b_purchase, p4b_dates, p4b_revenues, p4b_costs, p4b_terminal, rate,
                       overhaul_cost=p4b_overhaul_cost, overhaul_date=p4b_overhaul_date)
print(f"NPV Property 4b: ${npv_p4b:,.2f}")

In [ ]:
# Question 6: What is the NPV of an investment in Property 3?
print(f"Q6: NPV Property 3 = ${npv_p3:,.2f}")

q6_options = {'A': 104239, 'B': 104240, 'C': 104241, 'D': 104242,
              'E': 104243, 'F': 104244, 'G': 104245, 'H': 104246, 'I': 104247}
q6_answer = min(q6_options.keys(), key=lambda k: abs(q6_options[k] - round(npv_p3)))
print(f"Q6 answer: {q6_answer}")

In [ ]:
# Question 7: NPV of 4a less NPV of 4b (if 4b > 4a, enter negative)
q7_diff = npv_p4a - npv_p4b
print(f"Q7: NPV(4a) - NPV(4b) = ${q7_diff:,.0f}")
q7_val = round(q7_diff)
q7_answer_str = f"{q7_val:,}"
print(f"Q7 answer: {q7_answer_str}")

In [ ]:
# Question 8: Which property is highest in NPV value?
npv_dict = {'A': npv_p1, 'B': npv_p2, 'C': npv_p3, 'D': npv_p4a, 'E': npv_p4b}
for k, v in npv_dict.items():
    print(f"  {k}: ${v:,.2f}")
q8_answer = max(npv_dict.keys(), key=lambda k: npv_dict[k])
print(f"Q8 answer: {q8_answer}")

In [ ]:
# Question 9: Which properties should the company invest in to maximize NPV
# subject to total purchase price <= $1,700,000?
# Note: Can't invest in both 4a and 4b.
# Overhaul cost for 4b does NOT count toward purchase price constraint.

from itertools import combinations

properties = {
    '1': {'purchase': 450000, 'npv': npv_p1},
    '2': {'purchase': 550000, 'npv': npv_p2},
    '3': {'purchase': 500000, 'npv': npv_p3},
    '4a': {'purchase': 470000, 'npv': npv_p4a},
    '4b': {'purchase': 470000, 'npv': npv_p4b},
}

max_purchase = 1700000
best_npv = -float('inf')
best_portfolio = None

# Generate all valid combinations
prop_keys = list(properties.keys())
for r in range(1, len(prop_keys) + 1):
    for combo in combinations(prop_keys, r):
        # Can't have both 4a and 4b
        if '4a' in combo and '4b' in combo:
            continue
        total_purchase = sum(properties[p]['purchase'] for p in combo)
        if total_purchase > max_purchase:
            continue
        total_npv = sum(properties[p]['npv'] for p in combo)
        if total_npv > best_npv:
            best_npv = total_npv
            best_portfolio = combo

print(f"Best portfolio: {best_portfolio}")
print(f"Total purchase: ${sum(properties[p]['purchase'] for p in best_portfolio):,}")
print(f"Total NPV: ${best_npv:,.2f}")

q9_answer = ','.join(best_portfolio)
print(f"Q9 answer: {q9_answer}")

In [ ]:
# Question 10: If cost of capital was 4%, what is the maximum portfolio NPV?
rate_4pct = 0.04

npv_p1_4 = compute_npv(p1_purchase, p1_dates, p1_revenues, p1_costs, p1_terminal, rate_4pct)
npv_p2_4 = compute_npv(p2_purchase, p2_dates, p2_revenues, p2_costs, p2_terminal, rate_4pct)
npv_p3_4 = compute_npv(p3_purchase, p3_dates, p3_revenues, p3_costs, p3_terminal, rate_4pct)
npv_p4a_4 = compute_npv(p4a_purchase, p4a_dates, p4a_revenues, p4a_costs, p4a_terminal, rate_4pct)
npv_p4b_4 = compute_npv(p4b_purchase, p4b_dates, p4b_revenues, p4b_costs, p4b_terminal, rate_4pct,
                          overhaul_cost=p4b_overhaul_cost, overhaul_date=p4b_overhaul_date)

properties_4 = {
    '1': {'purchase': 450000, 'npv': npv_p1_4},
    '2': {'purchase': 550000, 'npv': npv_p2_4},
    '3': {'purchase': 500000, 'npv': npv_p3_4},
    '4a': {'purchase': 470000, 'npv': npv_p4a_4},
    '4b': {'purchase': 470000, 'npv': npv_p4b_4},
}

print("NPVs at 4%:")
for k, v in properties_4.items():
    print(f"  {k}: ${v['npv']:,.2f}")

best_npv_4 = -float('inf')
best_portfolio_4 = None

for r in range(1, len(prop_keys) + 1):
    for combo in combinations(prop_keys, r):
        if '4a' in combo and '4b' in combo:
            continue
        total_purchase = sum(properties_4[p]['purchase'] for p in combo)
        if total_purchase > max_purchase:
            continue
        total_npv = sum(properties_4[p]['npv'] for p in combo)
        if total_npv > best_npv_4:
            best_npv_4 = total_npv
            best_portfolio_4 = combo

print(f"Best portfolio at 4%: {best_portfolio_4}")
print(f"Total NPV at 4%: ${best_npv_4:,.2f}")

q10_val = round(best_npv_4)
q10_answer = f"{q10_val:,}"
print(f"Q10 answer: {q10_answer}")

In [ ]:
# Question 11: If revenues for all properties were paid monthly,
# what would be the total NPV (at 8%) of Properties 1, 2 and 3?

# Property 1 already pays monthly - no change
# Property 2: convert quarterly to monthly (same indexed annual amount, just spread monthly)
# Property 3: convert quarterly to monthly

# Property 2 with monthly revenue
p2_monthly_rev_base = p2_annual_revenue / 12.0
p2m_revenues = []
for d in p2_dates:
    factor = index_factor(d, p2_index_rate)
    rev = p2_monthly_rev_base * factor
    p2m_revenues.append(rev)

npv_p2m = compute_npv(p2_purchase, p2_dates, p2m_revenues, p2_costs, p2_terminal, rate)
print(f"NPV Property 2 (monthly rev): ${npv_p2m:,.2f}")

# Property 3 with monthly revenue
p3_monthly_rev_base = p3_annual_revenue / 12.0
p3m_revenues = []
for d in p3_dates:
    factor = index_factor(d, p3_index_rate)
    rev = p3_monthly_rev_base * factor
    p3m_revenues.append(rev)

npv_p3m = compute_npv(p3_purchase, p3_dates, p3m_revenues, p3_costs, p3_terminal, rate)
print(f"NPV Property 3 (monthly rev): ${npv_p3m:,.2f}")

# Total NPV of P1 + P2(monthly) + P3(monthly)
total_npv_q11 = npv_p1 + npv_p2m + npv_p3m
print(f"Q11: Total NPV P1+P2+P3 (monthly) = ${total_npv_q11:,.2f}")

q11_options = {'A': 268662, 'B': 268663, 'C': 268664, 'D': 268665,
               'E': 268666, 'F': 268667, 'G': 268668, 'H': 268669, 'I': 268670}
q11_answer = min(q11_options.keys(), key=lambda k: abs(q11_options[k] - round(total_npv_q11)))
print(f"Q11 answer: {q11_answer}")

In [ ]:
# Question 12: NPV (at 8%) of operating costs for Property 3 if
# first payment each year in June, second payment in December?
# (Instead of April and October)

# P3 costs: $1,000 first payment, $3,000 second payment, indexed at 2%
# Changed to: $1,000 in June, $3,000 in December

p3_alt_cost_dates = []
p3_alt_cost_cfs = []

for d in p3_dates:
    factor = index_factor(d, p3_index_rate)
    if d.month == 6:
        cost = 1000.0 * factor
    elif d.month == 12:
        cost = 3000.0 * factor
    else:
        cost = 0.0
    p3_alt_cost_dates.append(d)
    p3_alt_cost_cfs.append(cost)

# NPV of just the operating costs (positive values, discounted)
# We include the purchase date as day-0 for XNPV calculation
all_dates_q12 = [purchase_date] + p3_alt_cost_dates
all_cfs_q12 = [0.0] + p3_alt_cost_cfs  # costs are positive outflows but we compute NPV of costs

npv_p3_alt_costs = xnpv(rate, all_cfs_q12, all_dates_q12)
print(f"Q12: NPV of P3 alt operating costs = ${npv_p3_alt_costs:,.2f}")

q12_options = {'A': 19541, 'B': 19542, 'C': 19543, 'D': 19544,
               'E': 19545, 'F': 19546, 'G': 19547, 'H': 19548, 'I': 19549}
q12_answer = min(q12_options.keys(), key=lambda k: abs(q12_options[k] - round(npv_p3_alt_costs)))
print(f"Q12 answer: {q12_answer}")

In [ ]:
# ============================================================
# Final Answers
# ============================================================
answers = {
    "question1": q1_answer,
    "question2": q2_answer,
    "question3": q3_answer,
    "question4": q4_answer,
    "question5": q5_answer,
    "question6": q6_answer,
    "question7": q7_answer_str,
    "question8": q8_answer,
    "question9": q9_answer,
    "question10": q10_answer,
    "question11": q11_answer,
    "question12": q12_answer,
}

print("\nFinal Answers:")
for k, v in answers.items():
    print(f"  {k}: {v}")